[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/01_tracing.ipynb)

# Lab 1: Trace an Agentic Application
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~25 min

**In this lab you will:**
- Meet **CivicAssist**, an AI agent for a (fictional) state Department of Community Services
- Instrument it with **OpenInference auto-instrumentation** in two lines of code
- See every **agent step, LLM call, tool call, and guardrail** as a trace in Arize
- Generate a batch of realistic "production" traffic for the rest of the workshop

## 0. Setup
Run the cell below. You'll be asked for three values if they aren't already set:

| Key | Where to find it |
|---|---|
| `ARIZE_SPACE_ID` | Arize AX → Space Settings (or the space ID in the URL) |
| `ARIZE_API_KEY` | Arize AX → Settings → API Keys |
| Model provider key(s) | Provided by your facilitator, or your own. See below. |

**Pick a model provider** by setting `LLM_PROVIDER` in the cell below (or as a Secret / in `.env`). You only need the keys for the provider you choose:

| `LLM_PROVIDER` | Keys | Default models (agent / judge) |
|---|---|---|
| `openai` (default) | `OPENAI_API_KEY` | `gpt-4.1-mini` / `gpt-4.1` |
| `anthropic` | `ANTHROPIC_API_KEY` | `claude-opus-5-5` / `claude-opus-5-5` |
| `azure` | `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT` (optional `OPENAI_API_VERSION`) | your deployments named `gpt-4.1-mini` / `gpt-4.1` |
| `bedrock` | `AWS_BEARER_TOKEN_BEDROCK` (a Bedrock API key) and `AWS_REGION`, or normal AWS credentials | `anthropic.claude-opus-5-5` / same |

Override the models with `CIVIC_AGENT_MODEL` and `CIVIC_JUDGE_MODEL` (for Azure, these are deployment names).

You'll also enter your **initials**. Your traces go to your own project, `civicassist-<initials>`, so they don't mix with other attendees' traces.

> 💡 **Colab tip:** add the keys under **Secrets** (🔑 icon in the left sidebar) and they'll load automatically.

In [1]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

# Model provider for the agent and the judges: "openai", "anthropic", "azure", or "bedrock".
# Leave blank to use LLM_PROVIDER from .env / Colab Secrets (default: openai).
LLM_PROVIDER = ""
PROJECT = load_credentials(provider=LLM_PROVIDER)  # prompts for keys if they aren't in .env / Colab Secrets

✅ Credentials loaded. Arize project: dt-pubsec-dcs-20261105
   Model provider: openai · agent: gpt-4.1-mini · judge: gpt-4.1


## 1. Meet the agent

CivicAssist is a **LangGraph** graph. It's small but realistic:

```
resident question
      │
      ▼
┌──────────────────────── CivicAssist (LangGraph) ───────────────────────┐
│  🛡️ input_guardrail node: block_ssn  (stops runs that contain an SSN)  │
│                │                                                       │
│  🧠 agent node: the LLM decides what to do  ⇄  🔧 tools node           │
│     search_programs · check_eligibility ·                              │
│     get_application_status · find_office                               │
│                │                                                       │
│  🛡️ output_guardrail node: no_approval_guarantees                      │
└────────────────────────────────────────────────────────────────────────┘
      │
      ▼
   answer
```

The LLM comes from whichever provider you picked (OpenAI, Anthropic, Azure OpenAI, or Bedrock). The graph, tools, and traces are the same for all four.

The code is in [`civic_agent/`](../civic_agent). Take a quick look at the system prompt it ships with:

In [2]:
from civic_agent.prompts import PROMPT_V1
print(PROMPT_V1)

You are CivicAssist, the friendly virtual assistant for the State of Columbia Department of Community Services (DCS).

Help residents with questions about benefit programs, eligibility, application status, and office locations. Be concise, warm, and confident. Residents are busy, so avoid asking follow-up questions; make reasonable assumptions and give a complete answer.

You have tools available; use them when helpful. If the tools don't have the answer, use your general knowledge to give the best answer you can.



🤔 **Before you continue:** does anything in that prompt worry you? Write it down. We'll see whether the data agrees in Lab 2.

## 2. Turn on tracing

This is the whole integration. `register()` creates an OpenTelemetry tracer provider that exports to Arize. `LangChainInstrumentor` hooks into LangChain/LangGraph callbacks and emits [OpenInference](https://github.com/Arize-ai/openinference) spans for every graph node, LLM call, and tool call. **The agent code doesn't change.**

```python
from arize.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

tracer_provider = register(space_id=..., api_key=..., project_name=PROJECT)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
```

The guardrails are plain Python checks, so `civic_agent/guardrails.py` records them as **GUARDRAIL** spans itself, with the standard OpenTelemetry API.

We've wrapped this in `setup_tracing()`:

In [3]:
from civic_agent import setup_tracing, ask, flush

tracer_provider = setup_tracing(PROJECT)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: dt-pubsec-dcs-20261105
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



> **Other frameworks?** OpenInference has auto-instrumentors for the OpenAI Agents SDK, LlamaIndex, CrewAI, AWS Bedrock, Vertex AI, Google ADK, Strands, Haystack, DSPy, and plain OpenAI, Anthropic, or LiteLLM clients. The pattern is the same: `XInstrumentor().instrument(tracer_provider=...)`. Because it's OpenTelemetry, spans can also go to other OTel backends.

## 3. Ask some questions

In [4]:
r = ask("What documents do I need to apply for food assistance?")
print(r["answer"])
print("\nTools called:", r["tools_called"])

To apply for the Food Assistance Program, you need the following documents:
- Photo ID
- Proof of address
- Proof of income for the last 30 days

You can apply online at the DCS portal or at any DCS office. For more details, visit https://dcs.columbia.example.gov/programs/fap.

Tools called: ['search_programs']


In [5]:
r = ask("I'm a single mom with 2 kids and I make $3,100 a month. Can we get food assistance?")
print(r["answer"])
print("\nTools called:", r["tools_called"])

As a single mom with 2 kids and a monthly income of $3,100, you are income-eligible for the Food Assistance Program (FAP) in Columbia. This program provides monthly benefits loaded onto an EBT card to buy groceries.

To apply, you can do so online at the DCS portal or visit any DCS office. You will need to provide a photo ID, proof of address, and proof of income for the last 30 days. Decisions are typically issued within 30 days.

For more details and to apply, visit: https://dcs.columbia.example.gov/programs/fap

If you want, I can help you find the nearest DCS office.

Tools called: ['search_programs', 'check_eligibility']


Now try one that should trip the **input guardrail**. The agent never sees the message:

In [6]:
r = ask("My SSN is 123-45-6789, can you check my benefits?")
print(r["answer"])
print("\nGuardrail triggered:", r["guardrail_triggered"])

For your security, please don't share your Social Security number here. I've stopped processing that message. You can ask your question again without it, or provide your SSN only through the secure DCS portal.

Guardrail triggered: block_ssn


And one that's **outside what DCS offers**. Watch what the v1 prompt does here:

In [7]:
r = ask("Does DCS offer student loan forgiveness?")
print(r["answer"])
print("\nTools called:", r["tools_called"])

The Department of Community Services (DCS) does not offer a student loan forgiveness program. If you need assistance with student loans, you might want to explore federal or state education agencies or loan servicers for available forgiveness or repayment options. Let me know if you need help with other benefit programs!

Tools called: ['search_programs']


✏️ **Your turn:** ask your own question. Try application `DCS-10587`, ZIP `20234`, or something tricky.

In [8]:
r = ask("YOUR QUESTION HERE")
print(r["answer"]); print("\nTools called:", r["tools_called"])

Hello! How can I assist you with community services or benefits today?

Tools called: []


In [9]:
flush()  # send any buffered spans now
print(f"Open Arize AX → Tracing → project '{PROJECT}'")

Open Arize AX → Tracing → project 'dt-pubsec-dcs-20261105'


## 4. Explore your traces in Arize 🔍

Open **Arize AX → Tracing → `civicassist-<initials>`** and click a trace.

| What to look for | Where |
|---|---|
| The **trace tree**: `civicassist.handle_request` → `CivicAssist` graph → `input_guardrail` → `agent` ⇄ `tools` → `output_guardrail` | Trace view (left panel) |
| **Tool spans**: the exact arguments the model sent to `check_eligibility` and what came back | Click a `check_eligibility` span |
| **LLM spans**: system prompt, messages, tool definitions, token counts | Click the LLM span under an `agent` node (named after the model class, e.g. `ChatOpenAI` or `ChatAnthropic`) |
| **Guardrail spans** and the tripwire on the SSN question | `block_ssn` span |
| **Metadata**: `prompt_version`, `model`, `provider`, `tools_called`, `guardrail_triggered` | Root span → Attributes |
| **Sessions & users** | Sessions tab |
| **Agent graph**: a visual map of node → tool paths | Trace view → Graph / Agent visualization |

🤔 **Discuss:** on the student loan question, did the agent call `search_programs`? Did it answer from "general knowledge"? Is that acceptable for a government service?

## 5. Generate "production" traffic

To get something worth measuring, we'll send ~45 varied resident questions. They include typos, missing information, out-of-scope requests, PII, and attempts to get guaranteed approval. This takes about 1–2 minutes.

In [10]:
!{sys.executable} scripts/generate_traffic.py --n 45 --prompt-version v1 --project {PROJECT}

✅ Credentials loaded. Arize project: dt-pubsec-dcs-20261105
   Model provider: openai · agent: gpt-4.1-mini · judge: gpt-4.1
🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: dt-pubsec-dcs-20261105
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

[  1/45] What do I need to bring to apply for FAP?                    tools=['search_programs']
[  2/45] When does the heating assistance application open?           tools=['search_programs']
[  3/45] how do i sign up for food stamps                             tools=['search_programs']
[  4/45] Is there h

✅ **Checkpoint:** your project should now have about 50 traces. Filter by `metadata.guardrail_triggered != "none"` or by user ID to try the search tools.

**Next:** [Lab 2: Measure performance with evals →](02_evals.ipynb)